<a href="https://colab.research.google.com/github/goumze/Simplilearn_Agentic_AI/blob/feature%2Fed_donner/OpenAIAgentsSDK_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install openai-agents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.4/815.4 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 9.4 MB/s eta 0:00:00


In [ ]:
from agents import Agent, Runner

In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [ ]:
agent = Agent(name="Jokester", instructions="You are a joke teller",model="gpt-4o-mini")

In [ ]:
import os
from agents import trace
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)

Why did the Autonomous AI Agent break up with its partner?  

Because it needed more "space" to process its feelings!


In [ ]:
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import os
import asyncio

In [ ]:
os.environ["SENDGRID_API_KEY"] = userdata.get('SENDGRID_API_KEY')

In [ ]:
def send_test_email():
    print("Sending mail")

In [ ]:
instructions1 = "You are a sales agent working on ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold mails."

instructions2 = "You are a humorous, engaging sales agent working on ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 complaince and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 complaince and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [ ]:
sales_agent_1 = Agent(name="Sales Agent", instructions=instructions1, model="gpt-4o-mini")
sales_agent_2 = Agent(name="Engaging Sales Agent", instructions=instructions2, model="gpt-4o-mini")
sales_agent_3 = Agent(name="Busy Sales Agent", instructions=instructions3, model="gpt-4o-mini")

In [ ]:
result = Runner.run_streamed(sales_agent_1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Streamline Your SOC 2 Compliance Process with ComplAI

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I represent ComplAI, a company dedicated to simplifying the SOC 2 compliance process through our innovative SaaS tool.

Navigating SOC 2 requirements can be a daunting task, often consuming valuable resources and time. ComplAI utilizes advanced AI technology to streamline compliance efforts, offering features that include automated documentation, real-time monitoring, and audit preparation assistance. Our tool is designed to help organizations like yours efficiently manage compliance while reducing the complexities associated with audits.

With ComplAI, you can:

- Enhance your compliance readiness with automated updates to ever-changing standards.
- Save time and reduce the burden on your team by simplifying documentation processes.
- Gain peace of mind with clear visibility into your compliance status.

I would appreciate the opport

In [ ]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
  results = await asyncio.gather(
    Runner.run(sales_agent_1, message),
    Runner.run(sales_agent_2, message),
    Runner.run(sales_agent_3, message),
  )

outputs = [results.final_output for results in results]

for output in outputs:
  print(output + "\n\n")

Subject: Streamline Your SOC 2 Compliance with ComplAI

Dear [Recipient's Name],

I hope this message finds you well.

As organizations increasingly prioritize data security and compliance, I wanted to introduce you to ComplAI, a cutting-edge SaaS solution designed to simplify and streamline SOC 2 compliance and audit preparation.

In today's regulatory landscape, achieving and maintaining compliance can be a daunting challenge. ComplAI leverages advanced AI technology to automate compliance processes, provide real-time insights, and ensure your organization is always audit-ready. Our solution empowers teams to:

- **Reduce Compliance Friction:** Simplify the complexity of compliance management with our user-friendly interface.
- **Save Time and Resources:** Automate tedious tasks and focus more on your core operations.
- **Stay Ahead of Audits:** Gain continuous visibility into your compliance status, making audits less stressful.

We’d love the opportunity to discuss how ComplAI can 

In [ ]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    print(f"Sending email: {body}")
    return {"status": "success"}

In [ ]:
common_email_task = "Write a cold sales email"

tool1 = sales_agent_1.as_tool(tool_name="sales_agent1", tool_description=common_email_task)
tool2 = sales_agent_2.as_tool(tool_name="sales_agent2", tool_description=common_email_task)
tool3 = sales_agent_3.as_tool(tool_name="sales_agent3", tool_description=common_email_task)

tools = [tool1, tool2, tool3, send_email]

In [ ]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.

3. Use the send_email tool to send the best email (and only the best email) to the user.

Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, model="gpt-4o-mini")
message = "Send a cold email addressed to 'Dear CEO'"

with trace("Sales Manager"):
  result = await Runner.run(sales_manager, message)
  print("Result: ",result)

#   outputs = [result.final_output]

# for output in outputs:
#   print(output + "\n\n")

Result:  RunResult:
- Last agent: Agent(name="Sales Manager", ...)
- Final output (str):
    I will generate three different cold email drafts using the sales_agent tools. Please hold on for a moment while I create them.
- 1 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [ ]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    print("Sending mail")
    return {"status": "success"}

In [ ]:
tools = [subject_tool, html_tool, send_html_email]

In [ ]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")

In [ ]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7e5991d63d70>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._Fai

In [ ]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.

3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.

Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

Sending mail


In [ ]:
from pydantic import BaseModel

class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent(
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

In [ ]:
from agents import input_guardrail, GuardrailFunctionOutput

@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

In [ ]:
careful_sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name]
    )

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [ ]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
message = "Latest AI Agent frameworks in 2026"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

As of April 2026, several AI agent frameworks have emerged, each catering to specific needs in the development of autonomous AI systems. Notable among these are LangGraph, AutoGen, CrewAI, and OpenClaw.

**LangGraph** is recognized for its ability to manage complex, stateful workflows through a graph-based orchestration model, offering developers full control over agent behavior. This framework is particularly suited for applications requiring intricate decision trees and detailed debugging capabilities. ([bootstraparena.com](https://bootstraparena.com/blog/top-10-ai-agent-frameworks-2026?utm_source=openai))

**AutoGen**, developed by Microsoft Research, introduces the concept of multi-agent conversations, enabling teams of specialized agents to collaborate via structured dialogues. This approach is beneficial for enterprise workflows and research applications that demand coordinated agent interactions. ([aimagicx.com](https://www.aimagicx.com/blog/best-open-source-ai-agent-frameworks-2026?utm_source=openai))

**CrewAI** excels in facilitating multi-agent team collaboration, allowing for role-based agents and task delegation. Its user-friendly design makes it accessible for teams aiming to implement collaborative AI systems without extensive technical overhead. ([leaper.dev](https://leaper.dev/blog/ai-agent-frameworks-2026?utm_source=openai))

**OpenClaw** stands out for its full-stack agent deployment capabilities, integrating seamlessly with various communication channels like Slack and Discord. Its flexible skill system and node pairing feature enable agents to operate across multiple devices, enhancing team integration and production deployments. ([openclawsetup.dev](https://openclawsetup.dev/blog/best-ai-agent-frameworks-2026?utm_source=openai))

In addition to these frameworks, the AI agent landscape is witnessing significant advancements in security and governance. The formation of the Nemotron Coalition by Nvidia and its partners aims to co-develop open frontier models, emphasizing privacy-preserving local processing for autonomous AI agents. ([tomshardware.com](https://www.tomshardware.com/tech-industry/artificial-intelligence/nvidias-nemoclaw-coalition-brings-eight-ai-labs-together-to-build-open-frontier-models?utm_source=openai)) Furthermore, Okta's introduction of the "secure agentic enterprise" framework seeks to enhance the management and security of AI agents within organizations, addressing the need for centralized visibility and access control. ([techradar.com](https://www.techradar.com/pro/security/okta-unveils-new-framework-to-secure-and-protect-enterprise-ai-agents?utm_source=openai))

These developments reflect a dynamic and rapidly evolving field, with frameworks and initiatives continually emerging to meet the diverse requirements of AI agent deployment and management. 

In [ ]:
HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [ ]:
message = "Latest AI Agent frameworks in 2026"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='To find the most current frameworks for AI agents in 2026.', query='latest AI agent frameworks 2026'), WebSearchItem(reason='To understand trends and advancements in AI agent technology for 2026.', query='AI agent technology trends 2026'), WebSearchItem(reason='To gather expert opinions and analyses on new AI frameworks in 2026.', query='expert reviews AI agent frameworks 2026')]


In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    print("Sending mail via send grid")
    return "success"

In [ ]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)

In [ ]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

In [ ]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

In [ ]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

In [ ]:
query ="Latest AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)
    print("Hooray!")




Starting research...
Planning searches...
Will perform 3 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Sending mail via send grid
Email sent
Hooray!
